# Diffraction reflection angles (Bernina, `you` geometry)

Compute diffractometer angles for a given `(h,k,l)` reflection, or for a
whole list of reflections at once. Uses diffcalc's `you` UB geometry
(`mu, eta, chi, phi, gamma, delta`) &mdash; note this beamline's convention
calls the out-of-plane detector circle **gamma**, where diffcalc itself
calls it `nu`; that rename happens automatically everywhere below.

Fix any subset of the constraint angles below (checkbox) and diffcalc solves
for the rest. Unfixed reflections may have more than one valid solution
branch (shown as separate columns `sol. 0`, `sol. 1`, ...).

In [ ]:
import sys
from pathlib import Path

try:
    import bernina_prep  # noqa: F401
except ImportError:
    sys.path.insert(0, str(Path.cwd().parent))

import ipywidgets as widgets
from IPython.display import display

from bernina_prep.diffraction import build_ubcalc, make_constraints, solve_hkl

UC_W = "150px"
ANG_W = "150px"
HKL_W = "150px"

LAT_NAME = "custom lattice"


def calc_angles(
    uc_a=5.43, uc_b=5.43, uc_c=5.43,
    uc_alpha=90, uc_beta=90, uc_gamma=90,
    surface_normal="(1,1,1)",
    orientations=("(1,1,1) > (0,0,1)", "(1,1,-1) > (1,0,0)"),
    h=1, k=1, l=1,
    delta=0, delta_fixed=False,
    gam=0, gam_fixed=False,
    qaz=0, qaz_fixed=False,
    naz=0, naz_fixed=False,
    alpha=0, alpha_fixed=False,
    beta=0, beta_fixed=False,
    psi=0, psi_fixed=False,
    a_eq_b=0, a_eq_b_fixed=False,
    mu=0, mu_fixed=False,
    eta=0, eta_fixed=True,
    chi=0, chi_fixed=True,
    phi=0, phi_fixed=True,
    mu_is_gam=0, mu_is_gam_fixed=False,
    energy=8000,
):
    try:
        ubcalc = build_ubcalc(
            LAT_NAME, uc_a, uc_b, uc_c, uc_alpha, uc_beta, uc_gamma,
            list(orientations), surface_normal,
        )
        names = ["delta", "gam", "qaz", "naz", "alpha", "beta", "psi", "a_eq_b",
                 "mu", "eta", "chi", "phi", "mu_is_gam"]
        values = [delta, gam, qaz, naz, alpha, beta, psi, a_eq_b, mu, eta, chi, phi, mu_is_gam]
        fixed = [delta_fixed, gam_fixed, qaz_fixed, naz_fixed, alpha_fixed, beta_fixed,
                 psi_fixed, a_eq_b_fixed, mu_fixed, eta_fixed, chi_fixed, phi_fixed, mu_is_gam_fixed]
        cons = make_constraints({n: v for n, f, v in zip(names, fixed, values) if f})
        result = solve_hkl(ubcalc, cons, h, k, l, energy)
    except Exception as e:
        print(str(e))
        return
    display(result)
    global last_ubcalc, last_cons
    last_ubcalc, last_cons = ubcalc, cons


# Every widget is built and named explicitly, then wired to calc_angles by
# name via interactive_output. Deliberately NOT using widgets.interactive()
# here: it auto-generates extra widgets for any function parameter without
# an explicit override (e.g. from a string default), which silently shifts
# positional widget/argument pairings if you forget one - exactly the kind
# of bug we want this layout immune to.
ctrl = {
    "uc_a": widgets.FloatText(description="a$_{uc}$", value=5.43, layout=widgets.Layout(max_width=UC_W)),
    "uc_b": widgets.FloatText(description="b$_{uc}$", value=5.43, layout=widgets.Layout(max_width=UC_W)),
    "uc_c": widgets.FloatText(description="c$_{uc}$", value=5.43, layout=widgets.Layout(max_width=UC_W)),
    "uc_alpha": widgets.FloatText(description="$\\alpha_{uc}$", value=90, layout=widgets.Layout(max_width=UC_W)),
    "uc_beta": widgets.FloatText(description="$\\beta_{uc}$", value=90, layout=widgets.Layout(max_width=UC_W)),
    "uc_gamma": widgets.FloatText(description="$\\gamma_{uc}$", value=90, layout=widgets.Layout(max_width=UC_W)),
    "h": widgets.IntText(value=1, layout=widgets.Layout(max_width=HKL_W)),
    "k": widgets.IntText(value=1, layout=widgets.Layout(max_width=HKL_W)),
    "l": widgets.IntText(value=1, layout=widgets.Layout(max_width=HKL_W)),
    "surface_normal": widgets.Text(value="(1,1,1)", description="surf. normal"),
    "orientations": widgets.TagsInput(
        value=["(1,1,1) > (0,0,1)", "(1,1,-1) > (1,0,0)"], description="orientations"
    ),
}
angle_names = ["delta", "gam", "qaz", "naz", "alpha", "beta", "psi", "a_eq_b",
               "mu", "eta", "chi", "phi", "mu_is_gam"]
default_fixed = {"eta", "chi", "phi"}
for name in angle_names:
    if name == "a_eq_b":
        ctrl[name] = widgets.Checkbox(value=False, layout=widgets.Layout(max_width=ANG_W))
    elif name == "mu_is_gam":
        ctrl[name] = widgets.Checkbox(value=False, layout=widgets.Layout(max_width=ANG_W))
    else:
        ctrl[name] = widgets.FloatText(value=0, layout=widgets.Layout(max_width=ANG_W))
    ctrl[f"{name}_fixed"] = widgets.Checkbox(value=name in default_fixed, description="fix", indent=False)
ctrl["energy"] = widgets.FloatSlider(value=8000, min=1000, max=20000, description="energy [eV]")

output = widgets.interactive_output(calc_angles, ctrl)

ctr_uc = widgets.HBox(
    [ctrl["uc_a"], ctrl["uc_b"], ctrl["uc_c"], ctrl["uc_alpha"], ctrl["uc_beta"], ctrl["uc_gamma"]],
    layout=widgets.Layout(flex_flow="row wrap"),
)
ctr_hkl = widgets.HBox([ctrl["h"], ctrl["k"], ctrl["l"]])
ctr_or = widgets.VBox([ctrl["surface_normal"], ctrl["orientations"]])
ctr_ang = widgets.VBox(
    [widgets.HBox([widgets.Label(name, layout=widgets.Layout(width="70px")), ctrl[name], ctrl[f"{name}_fixed"]])
     for name in angle_names]
)
controls = widgets.VBox([ctr_uc, ctr_hkl, ctr_or, ctr_ang, ctrl["energy"]], layout=widgets.Layout(max_width="50%"))
display(widgets.HBox([controls, output]))


## Table of reflections

Reuses the crystal/orientation and fixed constraints set above. Enter one
`h,k,l` per line and pick which solution branch to report when a reflection
has more than one; reflections diffcalc can't solve are flagged in the
`error` column rather than aborting the whole table.

In [ ]:
from bernina_prep.diffraction import reflection_table

hkl_text = widgets.Textarea(
    value="1,0,0\n0,1,0\n0,0,1\n1,1,1",
    description="h,k,l list",
    layout=widgets.Layout(width="200px", height="120px"),
)
solution_idx = widgets.IntText(value=0, description="solution #")
compute_btn = widgets.Button(description="Compute table", button_style="primary")
table_out = widgets.Output()


def parse_hkl_lines(text):
    reflections = []
    for line in text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        h, k, l = (float(x) for x in line.replace(";", ",").split(","))
        reflections.append((h, k, l))
    return reflections


def on_compute(_):
    table_out.clear_output()
    with table_out:
        if "last_ubcalc" not in globals():
            print("Set up the crystal/orientation above first (run the single-reflection cell).")
            return
        reflections = parse_hkl_lines(hkl_text.value)
        display(reflection_table(last_ubcalc, last_cons, reflections, ctrl["energy"].value, solution=solution_idx.value))


compute_btn.on_click(on_compute)
display(widgets.VBox([widgets.HBox([hkl_text, solution_idx]), compute_btn, table_out]))


## Detector footprint (Q / hkl range)

For the currently-set crystal/orientation and a chosen `(h,k,l)`/solution
branch above, computes the range of reciprocal-lattice points a flat area
detector actually covers, given its distance from the sample (`tdet`) and
its physical size. Uses diffcalc's own `q_phi` convention directly (not a
second geometry library), extended from the single central ray to the
whole detector face, so it's consistent with the angles calculated above.

In [ ]:
from bernina_prep.diffraction import solve_hkl_positions, detector_hkl_grid, detector_hkl_summary

tdet_w = widgets.FloatText(value=500.0, description="tdet [mm]")
det_w_w = widgets.FloatText(value=77.0, description="width [mm]")
det_h_w = widgets.FloatText(value=77.0, description="height [mm]")
grid_n_w = widgets.IntText(value=9, description="grid n")
footprint_solution_idx = widgets.IntText(value=0, description="solution #")
footprint_btn = widgets.Button(description="Compute Q range", button_style="primary")
plot_btn = widgets.Button(description="Show 3D plot (slower)", button_style="")
footprint_out = widgets.Output()
plot_out = widgets.Output()

_last_footprint = {}


def _current_position():
    h, k, l = ctrl["h"].value, ctrl["k"].value, ctrl["l"].value
    energy = ctrl["energy"].value
    result = solve_hkl_positions(last_ubcalc, last_cons, h, k, l, energy)
    idx = footprint_solution_idx.value if footprint_solution_idx.value < len(result) else len(result) - 1
    pos, _ = result[idx]
    return pos, energy


def on_footprint(_):
    footprint_out.clear_output()
    with footprint_out:
        if "last_ubcalc" not in globals():
            print("Set up the crystal/orientation above first (run the single-reflection cell).")
            return
        pos, energy = _current_position()
        H, K, L = detector_hkl_grid(
            last_ubcalc, pos, energy, tdet_w.value, det_w_w.value, det_h_w.value, grid_n_w.value
        )
        _last_footprint.update(pos=pos, energy=energy, H=H, K=K, L=L)
        summary = detector_hkl_summary(
            last_ubcalc, pos, energy, tdet_w.value, det_w_w.value, det_h_w.value, grid_n_w.value
        )
        print(f"Detector center at delta={pos.delta:.3f}, gamma={pos.nu:.3f}, tdet={tdet_w.value:.1f} mm")
        display(summary.to_frame("value"))


def on_plot(_):
    plot_out.clear_output()
    with plot_out:
        if not _last_footprint:
            print("Compute the Q range first.")
            return
        import matplotlib.pyplot as plt

        H, K, L = _last_footprint["H"], _last_footprint["K"], _last_footprint["L"]
        fig = plt.figure(figsize=(6, 5))
        ax = fig.add_subplot(projection="3d")
        ax.plot_surface(H, K, L, alpha=0.6)
        ax.set_xlabel("h")
        ax.set_ylabel("k")
        ax.set_zlabel("l")
        n = H.shape[0] // 2
        cx, cy, cz = H[n, n], K[n, n], L[n, n]
        arrowlen = 0.3 * max(H.max() - H.min(), K.max() - K.min(), L.max() - L.min(), 1e-6)
        ax.quiver(cx, cy, cz, arrowlen, 0, 0, color="r")
        ax.quiver(cx, cy, cz, 0, arrowlen, 0, color="g")
        ax.quiver(cx, cy, cz, 0, 0, arrowlen, color="b")
        plt.show()


footprint_btn.on_click(on_footprint)
plot_btn.on_click(on_plot)
display(
    widgets.VBox(
        [
            widgets.HBox([tdet_w, det_w_w, det_h_w, grid_n_w, footprint_solution_idx]),
            widgets.HBox([footprint_btn, plot_btn]),
            footprint_out,
            plot_out,
        ]
    )
)
